# Analyze `minimum_atw` Chunked Output

This notebook inspects the parquet files generated by the chunked antibody-antigen example.

In [1]:
from pathlib import Path
import json

import pandas as pd

OUT_DIR = Path('/home/eva/minimum_atomworks/out_antibody_antigen_chunked')
OUT_DIR

PosixPath('/home/eva/minimum_atomworks/out_antibody_antigen_chunked')

In [2]:
tables = {
    'structures': pd.read_parquet(OUT_DIR / 'structures.parquet'),
    'chains': pd.read_parquet(OUT_DIR / 'chains.parquet'),
    'roles': pd.read_parquet(OUT_DIR / 'roles.parquet'),
    'interfaces': pd.read_parquet(OUT_DIR / 'interfaces.parquet'),
    'plugin_status': pd.read_parquet(OUT_DIR / 'plugin_status.parquet'),
    'bad_files': pd.read_parquet(OUT_DIR / 'bad_files.parquet'),
}

summary = pd.DataFrame(
    [
        {
            'table': name,
            'rows': len(df),
            'cols': len(df.columns),
            'columns': ', '.join(df.columns[:12]),
        }
        for name, df in tables.items()
    ]
)
summary

,table,rows,cols,columns
0,structures,10,8,"path, assembly_id, center__centroid_x, center_..."
1,chains,30,9,"path, assembly_id, chain_id, chstat__n_residue..."
2,roles,40,14,"path, assembly_id, role, id__n_atoms, rolseq__..."
3,interfaces,10,12,"path, assembly_id, pair, role_left, role_right..."
4,plugin_status,60,5,"path, assembly_id, plugin, status, message"
5,bad_files,0,2,"path, error"


In [3]:
tables['plugin_status'].groupby(['plugin', 'status'], dropna=False).size().rename('n').reset_index().sort_values(['plugin', 'status'])

,plugin,status,n
0,center_on_origin,ok,10
1,chain_stats,ok,10
2,identity,ok,10
3,interface_contacts,ok,10
4,role_sequences,ok,10
5,role_stats,ok,10


In [4]:
tables['structures'].head()

,path,assembly_id,center__centroid_x,center__centroid_y,center__centroid_z,id__n_atoms_total,id__n_chains,id__has_nan_coord
0,/home/eva/20260202_atomworks_plug_play/atw_pp/...,1,0.885804,-0.333535,-0.952488,4103,3,False
1,/home/eva/20260202_atomworks_plug_play/atw_pp/...,1,0.838313,-0.438043,-0.998483,4103,3,False
2,/home/eva/20260202_atomworks_plug_play/atw_pp/...,1,0.784270,-0.330999,-1.042894,4103,3,False
3,/home/eva/20260202_atomworks_plug_play/atw_pp/...,1,0.786609,0.592466,-1.633959,4103,3,False
4,/home/eva/20260202_atomworks_plug_play/atw_pp/...,1,-1.155203,0.629957,-1.042058,4103,3,False


In [5]:
tables['roles'][[
    'path',
    'role',
    'id__n_atoms',
    'rolseq__chain_ids',
    'rolseq__n_chains',
    'rolseq__sequence_length_total',
    'rolstat__n_residues',
]].head(10)

,path,role,id__n_atoms,rolseq__chain_ids,rolseq__n_chains,rolseq__sequence_length_total,rolstat__n_residues
0,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antibody,3197,B;C,2,421,212
1,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antigen,906,A,1,115,115
2,/home/eva/20260202_atomworks_plug_play/atw_pp/...,vh,1563,C,1,209,209
3,/home/eva/20260202_atomworks_plug_play/atw_pp/...,vl,1634,B,1,212,212
4,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antibody,3197,B;C,2,421,212
5,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antigen,906,A,1,115,115
6,/home/eva/20260202_atomworks_plug_play/atw_pp/...,vh,1563,C,1,209,209
7,/home/eva/20260202_atomworks_plug_play/atw_pp/...,vl,1634,B,1,212,212
8,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antibody,3197,B;C,2,421,212
9,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antigen,906,A,1,115,115


In [6]:
tables['interfaces'][[
    'path',
    'pair',
    'iface__contact_distance',
    'iface__n_contact_atom_pairs',
    'iface__n_left_interface_residues',
    'iface__n_right_interface_residues',
]].head(10)

,path,pair,iface__contact_distance,iface__n_contact_atom_pairs,iface__n_left_interface_residues,iface__n_right_interface_residues
0,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antibody__antigen,5.0,743,32,31
1,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antibody__antigen,5.0,722,32,32
2,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antibody__antigen,5.0,680,31,31
3,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antibody__antigen,5.0,602,32,29
4,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antibody__antigen,5.0,689,33,33
5,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antibody__antigen,5.0,576,30,30
6,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antibody__antigen,5.0,546,28,28
7,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antibody__antigen,5.0,616,27,29
8,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antibody__antigen,5.0,531,23,21
9,/home/eva/20260202_atomworks_plug_play/atw_pp/...,antibody__antigen,5.0,547,20,27


In [7]:
analysis_dir = OUT_DIR / 'dataset_analysis'
dataset_annotations = pd.read_parquet(analysis_dir / 'dataset_annotations.parquet')
interface_summary = pd.read_parquet(analysis_dir / 'interface_summary.parquet')
summary_json = json.loads((analysis_dir / 'summary.json').read_text())

dataset_annotations, interface_summary, summary_json

(                   key                     value   source
 0     n_interface_rows                        10  derived
 1  n_unique_structures                        10  derived
 2          n_role_rows                        40  derived
 3         dataset_name  antibody_antigen_chunked   config
 4       execution_mode               run-chunked   config
 5             modality          antibody_antigen   config
 6              project         atomworks_example   config
 7               source      atw_pp default input   config,
                 pair  n_rows  n_unique_paths  mean_contact_atom_pairs  \
 0  antibody__antigen      10              10                    625.2   
 
    mean_left_interface_residues  mean_right_interface_residues  
 0                          28.8                           29.1  ,
 {'dataset_analyses': 'dataset_annotations,interface_summary',
  'dataset_annotations_output': '/home/eva/minimum_atomworks/out_antibody_antigen_chunked/dataset_analysis/dataset_annotat

In [8]:
if not tables['bad_files'].empty:
    display(tables['bad_files'])
else:
    print('No bad files recorded.')

No bad files recorded.
